# Module 2 -- Step 6: Spend Tracking and Administration

In this step we explore the spend tracking and administration capabilities of the LLM Gateway.
LiteLLM tracks all token usage and cost per request, giving you full visibility into LLM spend.

In [ ]:
import json, requests

with open(".state.json") as f:
    state = json.load(f)

LLM_GATEWAY_URL = state["LLM_GATEWAY_URL"]
API_KEY = state["API_KEY"]
ADMIN_KEY = state["ADMIN_KEY"]
ADMIN_HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {ADMIN_KEY}"}

## Spend Logs

LiteLLM automatically tracks every request: model used, token counts, and estimated cost.
The `/spend/logs` endpoint returns this data so you can build cost dashboards and set budget alerts.

In [ ]:
resp = requests.get(f"{LLM_GATEWAY_URL}/spend/logs", headers=ADMIN_HEADERS, timeout=10)
logs = resp.json() if isinstance(resp.json(), list) else resp.json().get("data", [])

print(f"Total requests logged: {len(logs)}")
total_spend = sum(log.get("spend", 0) for log in logs)
print(f"Total spend: ${total_spend:.6f}\n")

for log in logs[:10]:
    print(f"  model={log.get('model', '?'):20s}  tokens={log.get('total_tokens', 0):6d}  spend=${log.get('spend', 0):.6f}")

## Virtual Key Info

Each virtual key has metadata including its name, team, budget, current spend, and allowed models.
Use the `/key/info` endpoint to inspect any key.

In [ ]:
resp = requests.get(
    f"{LLM_GATEWAY_URL}/key/info",
    headers=ADMIN_HEADERS,
    params={"key": API_KEY},
    timeout=10,
)
info = resp.json()
key_info = info.get("info", {})

print(f"Key Name:    {key_info.get('key_name', 'N/A')}")
print(f"Team ID:     {key_info.get('team_id', 'N/A')}")
print(f"Max Budget:  ${key_info.get('max_budget', 'N/A')}")
print(f"Spend:       ${key_info.get('spend', 0):.6f}")
print(f"Models:      {key_info.get('models', [])}")

## Proxy Health

LiteLLM's deep `/health` endpoint probes every backend model synchronously and routinely takes longer than API Gateway's 30-second integration timeout — through the Workshop gateway it returns `503 Service Unavailable`. Instead we use two fast endpoints: `/health/readiness` (proxy process + DB) and `/v1/models` (registered model catalog). For a full backend-connectivity probe, open the **Admin UI → Health** tab.

In [ ]:
resp = requests.get(f"{LLM_GATEWAY_URL}/health/readiness", timeout=10)
readiness = resp.json()
print(f"Proxy status: {readiness.get('status')}")
print(f"DB:           {readiness.get('db')}")
print(f"Version:      {readiness.get('litellm_version')}")

resp = requests.get(f"{LLM_GATEWAY_URL}/v1/models", headers=ADMIN_HEADERS, timeout=10)
models = resp.json().get("data", [])
print(f"\nRegistered models: {len(models)}")
for m in models[:10]:
    print(f"  - {m.get('id')}")
if len(models) > 10:
    print(f"  ... and {len(models) - 10} more")

## Admin UI

LiteLLM ships with a built-in admin dashboard that provides a visual interface
for managing keys, viewing spend, and monitoring model health.

In [ ]:
print(f"LiteLLM Admin UI: {LLM_GATEWAY_URL}/ui")
print("\nIn the Admin UI, navigate to the **Usage** tab to view spend per key and per model.")

---

**Next Step:** Proceed to [Step 7: Cleanup](step-7-cleanup.ipynb).